# Comparison Notebook – Streaming + 3 Modelle

Dieses Notebook lädt **drei trainierte Modelle** und wendet sie auf denselben Kafka-Stream an.  
So könnt ihr die Modelle **fair vergleichen**, weil jedes Modell **genau dieselben Events** sieht.

## Ziel
- Kafka-Events aus `transactions_raw` lesen
- Dasselbe Event an alle 3 Modelle senden
- Ergebnisse nebeneinander ausgeben
- Optional einfache Live-Metriken sammeln

## Erwartete Modelle
- `models/isolation_forest.joblib`
- `models/random_forest.joblib`
- `models/logistic_regression.joblib`

> Falls eure Dateinamen anders sind, passt sie unten in der Konfigurationszelle an.

In [6]:
# Falls nötig: einmalig installieren
# !pip install kafka-python joblib pandas scikit-learn

In [7]:
import json
import os
import joblib
import numpy as np
import pandas as pd

from kafka import KafkaConsumer
from sklearn.metrics import confusion_matrix, classification_report

## Konfiguration

Hier legst du fest:
- Kafka Topic
- Consumer Group
- Wie viele Events gelesen werden sollen
- Wo die Modell-Dateien liegen

In [8]:
KAFKA_TOPIC = "transactions_raw"
KAFKA_BOOTSTRAP = "localhost:9092"
CONSUMER_GROUP = "comparison-consumer-group-v1"

# Begrenzung für Demo / Tests
MAX_MESSAGES = 10000

# Pfade zu den drei Modellen
ISOLATION_FOREST_PATH = "models/isolation_forest.joblib"

# ------------------------------------------------------------
# HIER die Modell-Datei von Kollege 1 eintragen
# Beispiel: Random Forest Modell
# ------------------------------------------------------------
RANDOM_FOREST_PATH = "models/random_forest.joblib"

# ------------------------------------------------------------
# HIER die Modell-Datei von Kollege 2 eintragen
# Beispiel: Logistic Regression Modell
# ------------------------------------------------------------
LOGISTIC_REGRESSION_PATH = "models/logistic_regression.joblib"

## Modelle laden

Falls ein Modell noch nicht vorhanden ist, bekommt ihr hier direkt eine Fehlermeldung.

In [9]:
assert os.path.exists(ISOLATION_FOREST_PATH), f"Datei nicht gefunden: {ISOLATION_FOREST_PATH}"

# ------------------------------------------------------------
# Diese beiden Assertions prüfen die Modelle deiner Kollegen.
# Falls die Dateien noch nicht existieren, zuerst deren Notebooks
# ausführen und die Modelle als .joblib speichern.
# ------------------------------------------------------------

#assert os.path.exists(RANDOM_FOREST_PATH), f"Datei nicht gefunden: {RANDOM_FOREST_PATH}"
assert os.path.exists(LOGISTIC_REGRESSION_PATH), f"Datei nicht gefunden: {LOGISTIC_REGRESSION_PATH}"

isolation_forest_model = joblib.load(ISOLATION_FOREST_PATH)

# ------------------------------------------------------------
# HIER wird das Random-Forest-Modell deines Kollegen geladen.
# ------------------------------------------------------------
#random_forest_model = joblib.load(RANDOM_FOREST_PATH)

# ------------------------------------------------------------
# HIER wird das Logistic-Regression-Modell deines Kollegen geladen.
# ------------------------------------------------------------
logistic_regression_model = joblib.load(LOGISTIC_REGRESSION_PATH)

print("Alle Modelle wurden erfolgreich geladen.")

Alle Modelle wurden erfolgreich geladen.


## Kafka Consumer erstellen

Der Consumer liest aus demselben Topic wie euer bisheriger Receiver.

In [ ]:
consumer = KafkaConsumer(
    KAFKA_TOPIC,
    bootstrap_servers=KAFKA_BOOTSTRAP,
    auto_offset_reset="earliest",
    enable_auto_commit=True,
    group_id=CONSUMER_GROUP,
    value_deserializer=lambda m: json.loads(m.decode("utf-8"))
)

print("Kafka Consumer läuft und wartet auf Nachrichten ...")

## Hilfsfunktionen

Wir bringen die Events in ein gemeinsames Format, damit alle Modelle dieselben Eingaben bekommen.

In [16]:
FEATURE_COLUMNS = [f"V{i}" for i in range(1, 29)] + ["Amount"]

def event_to_feature_vector(event):
    '''
    Erwartetes Event-Format:
    {
      "event_id": ...,
      "event_time": ...,
      "event_time_real": ...,
      "amount": ...,
      "features": [V1 ... V28],
      "label": ...
    }
    '''

FEATURE_COLUMNS = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount"]

def event_to_feature_vector(event):
    time_value = event["event_time"]
    features = event["features"]
    amount = event["amount"]
    return pd.DataFrame([[time_value] + features + [amount]], columns=FEATURE_COLUMNS)

def get_actual_label(event):
    return int(event.get("label", 0))

## Vorhersage-Funktionen pro Modell

Hier wird für jedes Modell ein **einheitliches Ausgabeformat** erzeugt.

### Wichtiger Unterschied
- Isolation Forest: gibt `predict()` mit `1` (normal) und `-1` (Anomalie) zurück
- Random Forest / Logistic Regression: geben Klassenvorhersagen `0/1` zurück
- Optional kann bei RF/LR auch `predict_proba()` verwendet werden

In [17]:
def predict_isolation_forest(model, X):
    raw_pred = model.predict(X)[0]              # 1 = normal, -1 = anomaly
    score = float(model.decision_function(X)[0])
    pred = 1 if raw_pred == -1 else 0          # Umwandlung auf Fraud-Logik: 1 = Fraud/Anomalie
    return {
        "prediction": pred,
        "score": score
    }

def predict_supervised_model(model, X):
    pred = int(model.predict(X)[0])

    # Falls das Modell Wahrscheinlichkeiten unterstützt, geben wir sie mit aus
    if hasattr(model, "predict_proba"):
        proba = float(model.predict_proba(X)[0][1])
    else:
        proba = None

    return {
        "prediction": pred,
        "score": proba
    }

## Streaming-Vergleich starten

Diese Zelle:
- liest Events aus Kafka
- schickt jedes Event an alle 3 Modelle
- gibt die Ergebnisse strukturiert aus
- sammelt sie zusätzlich in einer Liste

In [18]:
results = []

count = 0

for message in consumer:
    event = message.value
    X = event_to_feature_vector(event)
    actual_label = get_actual_label(event)

    # Eigenes Modell: Isolation Forest
    iso_result = predict_isolation_forest(isolation_forest_model, X)

    # ------------------------------------------------------------
    # HIER wird das Random-Forest-Modell deines Kollegen verwendet.
    # Das Event X wird an das Modell übergeben.
    # ------------------------------------------------------------
    #rf_result = predict_supervised_model(random_forest_model, X)

    # ------------------------------------------------------------
    # HIER wird das Logistic-Regression-Modell deines Kollegen verwendet.
    # Das Event X wird an das Modell übergeben.
    # ------------------------------------------------------------
    lr_result = predict_supervised_model(logistic_regression_model, X)

    row = {
        "event_id": event["event_id"],
        "event_time": event["event_time"],
        "amount": event["amount"],
        "actual_label": actual_label,

        "iso_pred": iso_result["prediction"],
        "iso_score": iso_result["score"],

        #"rf_pred": rf_result["prediction"],
        #"rf_score": rf_result["score"],

        "lr_pred": lr_result["prediction"],
        "lr_score": lr_result["score"],
    }

    results.append(row)

    print(
        f"event_id={row['event_id']} | actual={row['actual_label']} | "
        f"IF(pred={row['iso_pred']}, score={row['iso_score']:.4f}) | "
        #f"RF(pred={row['rf_pred']}, score={row['rf_score']}) | "
        f"LR(pred={row['lr_pred']}, score={row['lr_score']})"
    )

    count += 1
    if count >= MAX_MESSAGES:
        break

consumer.close()
print(f"Fertig. {count} Events gelesen.")

event_id=0 | actual=0 | IF(pred=0, score=0.0906) | 
event_id=1 | actual=0 | IF(pred=0, score=0.1190) | 
event_id=2 | actual=0 | IF(pred=0, score=0.0176) | 
event_id=3 | actual=0 | IF(pred=0, score=0.0652) | 
event_id=4 | actual=0 | IF(pred=0, score=0.0964) | 
event_id=5 | actual=0 | IF(pred=0, score=0.1122) | 
event_id=6 | actual=0 | IF(pred=0, score=0.1071) | 
event_id=7 | actual=0 | IF(pred=0, score=0.0041) | 
event_id=8 | actual=0 | IF(pred=0, score=0.0713) | 
event_id=9 | actual=0 | IF(pred=0, score=0.1062) | 
event_id=10 | actual=0 | IF(pred=0, score=0.0875) | 
event_id=11 | actual=0 | IF(pred=0, score=0.0559) | 
event_id=12 | actual=0 | IF(pred=0, score=0.0935) | 
event_id=13 | actual=0 | IF(pred=0, score=0.0970) | 
event_id=14 | actual=0 | IF(pred=0, score=0.0323) | 
event_id=15 | actual=0 | IF(pred=0, score=0.0707) | 
event_id=16 | actual=0 | IF(pred=0, score=0.1080) | 
event_id=17 | actual=0 | IF(pred=0, score=0.0904) | 
event_id=18 | actual=0 | IF(pred=1, score=-0.0302) | 
ev

## Ergebnisse als DataFrame

Hier könnt ihr die Predictions direkt tabellarisch vergleichen.

In [21]:
results_df = pd.DataFrame(results)
results_df.head(2000)

,event_id,event_time,amount,actual_label,iso_pred,iso_score
0,0,0.0,149.62,0,0,0.090635
1,1,0.0,2.69,0,0,0.119047
2,2,1.0,378.66,0,0,0.017620
3,3,1.0,123.50,0,0,0.065170
4,4,2.0,69.99,0,0,0.096414
...,...,...,...,...,...,...
1995,1995,1534.0,104.49,0,0,0.061820
1996,1996,1535.0,1.00,0,0,0.091732
1997,1997,1540.0,10.98,0,0,0.108331
1998,1998,1542.0,13.08,0,0,0.112630


## Einfache Auswertung pro Modell

Damit bekommt ihr schnell einen Überblick, wie sich die drei Modelle auf denselben Streaming-Events verhalten.

In [20]:
def evaluate_model(results_df, pred_col, actual_col="actual_label", model_name="Model"):
    y_true = results_df[actual_col]
    y_pred = results_df[pred_col]

    print(f"=== {model_name} ===")
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print()
    print(classification_report(y_true, y_pred, zero_division=0))
    print("-" * 60)

evaluate_model(results_df, "iso_pred", model_name="Isolation Forest")
evaluate_model(results_df, "rf_pred", model_name="Random Forest")
evaluate_model(results_df, "lr_pred", model_name="Logistic Regression")

=== Isolation Forest ===
Confusion Matrix:
[[9387  575]
 [   2   36]]

              precision    recall  f1-score   support

           0       1.00      0.94      0.97      9962
           1       0.06      0.95      0.11        38

    accuracy                           0.94     10000
   macro avg       0.53      0.94      0.54     10000
weighted avg       1.00      0.94      0.97     10000

------------------------------------------------------------


KeyError: 'rf_pred'

## Hinweise

### 1. Einheitliches Feature-Format
Alle drei Modelle müssen mit denselben Eingaben trainiert worden sein:
- `V1` bis `V28`
- `Amount`

### 2. Unterschiedliche Scores
- **Isolation Forest**: Score ist ein Anomalie-Score (`decision_function`)
- **Random Forest / Logistic Regression**: Score ist, falls verfügbar, die Wahrscheinlichkeit für Klasse 1

### 3. Fairer Vergleich
Dieses Notebook ist nur dann ein fairer Vergleich, wenn:
- alle drei Modelle auf demselben Trainings-/Testkonzept basieren
- alle drei dasselbe Event-Format erwarten